# Information Retrieval Systems
## Phase 2

---
> Eleni Kechrioti p3210078@aueb.gr <br /> 
> Dejvid Isufaj p3210056@aueb.gr

### Introduction

This project focuses on the implementation and evaluation of an information retrieval system based on a text indexing approach. The goal is to efficiently search for relevant documents in response to specific queries, and to measure the system’s performance using the Mean Average Precision (MAP) metric.

For evaluation, the `trec_eval` tool was utilized, which allows comparison of the system’s retrieved results against a ground truth set of relevance judgments. This evaluation is essential for assessing and improving the accuracy and relevance of the search results provided to the user.

If you don't already have those installed, you should do it as we are going to use them below.

In [1]:
#!pip install nltk elasticsearch
!pip install gensim

   ---------------------------------------- 0.0/24.0 MB ? eta -:--:--
   - -------------------------------------- 1.0/24.0 MB 10.1 MB/s eta 0:00:03
   -------------------- ------------------- 12.6/24.0 MB 41.5 MB/s eta 0:00:01
   ------------------------------------ --- 22.0/24.0 MB 45.0 MB/s eta 0:00:01
   ---------------------------------------- 24.0/24.0 MB 36.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/45.9 MB ? eta -:--:--
   ----------- ---------------------------- 13.6/45.9 MB 65.9 MB/s eta 0:00:01
   ------------------- -------------------- 22.0/45.9 MB 53.6 MB/s eta 0:00:01
   --------------------------- ------------ 31.2/45.9 MB 49.5 MB/s eta 0:00:01
   ------------------------------------- -- 43.5/45.9 MB 52.2 MB/s eta 0:00:01
   ---------------------------------------  45.9/45.9 MB 53.0 MB/s eta 0:00:01
   ---------------------------------------- 45.9/45.9 MB 43.6 MB/s eta 0:00:00
  Attempting uninstall: scipy
    Found existing installation: scipy 1.

DEPRECATION: Loading egg at c:\python312\lib\site-packages\ember-0.1.0-py3.12.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330

[notice] A new release of pip is available: 25.0 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Setup and Imports

The following Python code sets up the necessary imports and downloads required NLTK resources for text preprocessing, as well as imports Elasticsearch libraries for indexing and bulk operations.

In [1]:
import string
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer
import json
from elasticsearch import Elasticsearch
from elasticsearch.helpers import bulk
import json
import gensim 
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess


# Download required NLTK datasets
nltk.download('stopwords')    # Common stopwords for filtering
nltk.download('punkt')        # Tokenizer models for splitting text
nltk.download('punkt_tab')    # Additional tokenizer data (tabs handling)
nltk.download('wordnet')      # WordNet lexical database for lemmatization
nltk.download('omw-1.4')      # Open Multilingual WordNet (needed for some lemmatization)

from nltk.corpus import wordnet as wn

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\eleni\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\eleni\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\eleni\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\eleni\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\eleni\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Instantiate an Elasticsearch client connected to the local server at port 9200. You should first run the elasticsearch.bat

In [3]:
client = Elasticsearch("http://localhost:9200")

We define a custom Elasticsearch index mapping with advanced settings:

- **Similarity:** We use the BM25 similarity algorithm given by ElasticSearch. While TF-IDF is a simple and intuitive method for weighting terms based on their frequency and rarity, BM25 offers an advanced and fine-tuned method that considers additional factors such as document length and frequency saturation.

- **Analysis:** We use the built-in English analyzer for both indexing (`default`) and searching (`default_search`). Also we removed `case specific stopwords` and stemmed the tokens using ElasticSearch's `porter stemmer`.

- **Mappings:** We specify fields like `title`, `text`, `authors`, `year`, `references`, and `cited_by` as text fields. The fields `title` and `text`, each with `analyzer: scidocs_analyzer`, `similarity: BM25`, and both copy their content to `allContent` to implement full text search. The field `allContent` also uses `scidocs_analyzer + BM25`, giving a unified field for “search across everything.”   

Finally, we create an Elasticsearch index called `corpus_index` with these customized settings and mappings.

In [92]:
custom_mapping = {
  "settings": {
      "analysis": {
            "filter": {
                "english_stop": {
                    "type": "stop",
                    "stopwords": "_english_"
                },
                "scidocs_stop": {
                    "type": "stop",
                    "stopwords": [
                        "the", "and", "of", "in", "to", "a", "is", "for", "on", "that",
                        "with", "as", "by", "an", "at", "be", "this", "from", "or", "are",
                        "was", "which", "it", "we", "study", "results", "method", "methods",
                        "figure", "table", "data", "analysis", "based", "using", "show",
                        "shown", "et", "al", "also", "these", "those", "their", "were",
                        "may", "can", "used", "use", "such", "have", "has", "had"
                    ]
                },
                "english_stemmer": {
                    "type": "stemmer",
                    "language": "english"
                }
            },
            "analyzer": {
                "my_analyzer": {
                    "tokenizer": "standard",
                    "filter": [
                        "lowercase",
                        "english_stemmer",
                        "english_stop"
                    ]
                }
            }
        }
    },
  "mappings": {
        "properties": {
            "title": {
                "type": "text",
                "similarity": "BM25",
                "analyzer": "my_analyzer",
                "copy_to": "allContent"
            },
            "text": {
                "type": "text",
                "similarity": "BM25",
                "analyzer": "my_analyzer",
                "copy_to": "allContent"
            },
            "authors": {
                "type": "text",
            },
            "year": {
                "type": "text",
            },
            "references": {
                "type": "text",
            },
            "cited_by": {
                "type": "text",
            },
            "allContent": {
                "type": "text",
                "similarity": "BM25",
                "analyzer": "my_analyzer"
            }
        }
    }
}


client.indices.create(index='corpus_index', body=custom_mapping)

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'corpus_index'})

In [91]:
client.indices.delete(index="corpus_index")

ObjectApiResponse({'acknowledged': True})

## Document Loading, Text Preprocessing, Preprocessing, and Bulk Indexing into Elasticsearch

### 1. Function `build_bulk_doc(raw_line: str)`

- Takes a raw JSON string (`raw_line`) representing a scientific document.
- Parses the JSON into a Python dictionary (`doc`).
- Extracts metadata from the document (`meta = doc["metadata"]`).
- Constructs a dictionary (`final_doc`) formatted for Elasticsearch bulk indexing, including fields such as:
  - `title`
  - `text`
  - `authors`
  - `year`
  - `references`
  - `cited_by`
- Tokenizes the document text using `simple_preprocess` (which lowercases, removes accents, and splits the text into tokens).
- If tokens exist, appends them to the global list `sentences` for later use in Word2Vec training.
- Returns the formatted document for bulk insertion into Elasticsearch.

### 2. Bulk Indexing into Elasticsearch

- Opens the corpus file `scidocs/corpus.jsonl`, which contains one JSON document per line.
- Uses the Elasticsearch Python client’s `bulk` helper to index all documents efficiently by streaming them through the `build_bulk_doc`.
- Sets `refresh=True` to make the newly indexed documents immediately searchable.

We construct a dictionary for each document with the required Elasticsearch indexing format, including the index name, document ID, and source fields.

Finally, we use Elasticsearch’s bulk API to efficiently index all documents into the `corpus_index` and confirm the total number of indexed documents.

In [93]:
sentences = []
def build_bulk_doc(raw_line: str, fields):
    doc = json.loads(raw_line)
    meta = doc["metadata"]
    final_doc = {
        '_index': 'corpus_index',
        '_id':    doc["_id"],
        '_source':{
            'title': doc['title'],
            'text': doc['text'],
            'authors': meta.get('authors', []),
            'year': meta.get('year'),
            'references': meta.get('references', []),
            'cited_by': meta.get('cited_by', 0)
        }
    }
    full_text = " ".join(str(doc[field]) for field in fields if field in doc)
    tokens = simple_preprocess(full_text, deacc=True)
    if tokens:
        sentences.append(tokens)
    tokens = simple_preprocess(doc['text'], deacc=True)
    if tokens:
        sentences.append(tokens)

    return final_doc


with open("scidocs/corpus.jsonl", "r", encoding="utf-8") as file:
    bulk(
        client,
        (build_bulk_doc(line, fields = ["title", "text", "authors", "year", "references", "cited_by"]) for line in file),
        refresh=True
    )



doc_count = client.count(index='corpus_index')["count"]
print(f"Indexed {doc_count:,} documents  in {'corpus_index'}")
print(f"Collected {len(sentences):,} tokenised sentences for word2vec.")

Indexed 25,657 documents  in corpus_index
Collected 50,964 tokenised sentences for word2vec.


## Search Results Formatting and Display

We define a function to neatly format and display the search results returned by Elasticsearch. If no results are found, we inform the user accordingly. Otherwise, for each document in the search hits, we extract relevant fields such as the document ID, relevance score, text content, title, authors, publication year, references, and citations. These details are then presented in a clear, readable format to facilitate quick understanding of each search result.

In [6]:
def pretty_search_response(response):
    if len(response["hits"]["hits"]) == 0:
        print("Your search returned no results.")
    else:
        for hit in response["hits"]["hits"]:
            id = hit["_id"]
            score = hit["_score"]
            text = hit["_source"]["text"]
            title = hit["_source"]["title"]
            authors = hit["_source"]["authors"]
            year = hit["_source"]["year"]
            references = hit["_source"]["references"]
            cited_by = hit["_source"]["cited_by"]
            
            pretty_output = f"\nID: {id}\nScore: {score}\nText: {text}\nTitle: {title}\nAuthors: {authors}\nYear: {year}\nReferences: {references}\nCited_by: {cited_by}\n"

            print(pretty_output)

## Training the Word2Vec Model

This code snippet trains a Word2Vec model using the tokenized sentences collected earlier.

- `sentences=sentences`: Uses the list of tokenized sentences gathered from the corpus as input data for training.
- `vector_size=250`: Sets the dimensionality of the word vectors to 250. Higher dimensions can capture more semantic nuances but require more data and computation.
- `window=4`: Defines the context window size, i.e., the number of words to consider before and after the target word during training.
- `min_count=4`: Ignores all words that appear fewer than 4 times in the corpus, filtering out rare words to reduce noise.
- `workers=4`: Utilizes 4 CPU cores for parallel training to speed up the process.
- `epochs=10`: Number of iterations over the entire dataset to improve learning.
- `sg=1`: Chooses the Skip-gram architecture (1 = skip-gram, 0 = CBOW). Skip-gram tends to perform better on smaller datasets and captures rare word representations more effectively.

After training, the script prints the size of the learned vocabulary (`len(model.wv)`), indicating how many unique words the model learned embeddings for.



In [ ]:
print("Training Word2Vec …")
model = Word2Vec(
    sentences=sentences,
    vector_size=250,
    window=4,
    min_count=4,
    workers= 4,
    epochs=10,
    sg=1   # 1 = skip-gram, 0 = CBOW
)
print("Word2Vec vocabulary:", len(model.wv))

Training Word2Vec …
Word2Vec vocabulary: 40054


In [104]:
model.build_vocab(sentences)
model.train(sentences, total_examples=model.corpus_count, epochs=25)

(170628778, 215519075)

### Configuration Variables

We set the key configuration variables for our evaluation process:

- **queries_file**: Path to the JSONL file containing the queries to be run against the index.
- **run_file**: Path where the search results will be saved in TREC run file format.
- **index_name**: Name of the Elasticsearch index we will query.
- **top_k**: Number of top results to retrieve for each query during search.


In [16]:
queries_file = "scidocs/queries.jsonl"
run_file = "trec_eval/my_results"
index_name = "corpus_index"
top_k = [20, 30, 50]

## Query Expansion with WordNet and Word2Vec

We leverage WordNet's POS tagging and Word2Vec's semantic similarity to enrich queries with both linguistically related and contextually similar words, improving the potential for retrieving relevant documents in an information retrieval system.

- **POS Tag Conversion:**  
  The function `get_wordnet_pos` converts NLTK's POS tags (like 'JJ' for adjective, 'NN' for noun) to the format expected by WordNet (`wn.ADJ`, `wn.NOUN`, etc.). This helps in accurately identifying synonyms.

- **Stop Words and Tokenization:**  
  The `expand_query_combined` function first tokenizes the input query into words, converting them to lowercase and removing punctuation and stopwords to focus on meaningful terms.

- **Expansion Logic:**  
  For each word in the query:  
  - It adds the original word to the expanded set.  
  - If the word exists in the Word2Vec vocabulary and has a valid POS tag, it finds the top similar words (neighbors) using Word2Vec.  
  - It adds neighbors whose similarity score exceeds the defined threshold (`sim_threshold`, default 0.75) to the expanded query.


In [96]:
from nltk import pos_tag
from nltk.corpus import stopwords
import string


# It turns POS tags from nltk(JJ, NN) in compatible format with Wordent (wn.ADJ, wn.NOUN, etc)
def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'):
        return wn.ADJ
    elif treebank_tag.startswith('V'):
        return wn.VERB
    elif treebank_tag.startswith('N'):
        return wn.NOUN
    elif treebank_tag.startswith('R'):
        return wn.ADV
    else:
        return None

def expand_query_combined(query, model, sim_threshold=0.75, topn=3):
    stop_words = set(stopwords.words('english') + list(string.punctuation))
    tokens = word_tokenize(query.lower())
    tagged = pos_tag(tokens)
    
    expanded = set()
    
    for word, tag in tagged:
        if word in stop_words:
            continue
        wn_tag = get_wordnet_pos(tag)
        expanded.add(word)

        if word in model.wv and wn_tag is not None:
            for neighbor, sim in model.wv.most_similar(word, topn=topn):
                if sim >= sim_threshold:
                    expanded.add(neighbor)
    
    return " ".join(expanded)


### Running Queries and Generating TREC Run File

We iterate over each query from the JSONL file and perform a search on the Elasticsearch index. For each query:

- Extract the query ID and query text.
- Perform a search on the specified index, retrieving the top *k* results.
- For each retrieved document, write an entry to the run file in TREC format, including:
  - Query ID
  - A constant "Q0" (required by TREC format)
  - Document ID
  - Rank of the document in the results
  - Relevance score (rounded to 4 decimals)
  - Run tag (to identify this experiment)

This run file will be used later for evaluation with `trec_eval`.

In [105]:
run_tag = "my_run"
for topk in top_k:
    file_path = run_file + str(topk) + ".run"
    with open(queries_file, "r", encoding="utf-8") as qf, open(file_path, "w", encoding="utf-8") as rf:
        for line in qf:
            query_obj = json.loads(line)
            query_id = query_obj["_id"]
            query_text = query_obj["text"]

            expanded_query_str = expand_query_combined(query_text, model)

            response = client.search(
                index=index_name,
                size=topk,
                query={
                    "bool": {
                        "should": [
                            {"match": { "title": {"query": query_text,"boost": 3}}},
                            {"match": {"text": {"query": query_text,"boost": 3}}},
                            {"match": {"allContent": {"query": query_text,"boost": 3}}},
                            {"match": { "title": {"query": expanded_query_str,"boost": 1}}},
                            {"match": {"text": {"query": expanded_query_str,"boost": 1}}},
                            {"match": {"allContent": {"query": expanded_query_str,"boost": 1}}}
                        ]
                    }
                }
            )

            for rank, hit in enumerate(response["hits"]["hits"], start=1):
                doc_id = hit["_id"]
                score = hit["_score"]
                rf.write(f"{query_id} Q0 {doc_id} {rank} {score:.4f} {run_tag}\n")


We run the `trec_eval` tool to evaluate the retrieval performance by computing the mean average precision (MAP) using our relevance judgments (`test_trec.tsv`) and the search results (`my_results.run`).

In [106]:
files = ['my_results20.run','my_results30.run','my_results50.run']
for f in files:
    print(f'\nTrec Eval Evaluation for file {f}\n')
    !.\trec_eval\trec_eval -m recall trec_eval/test_trec.tsv trec_eval/{f} 2>null


Trec Eval Evaluation for file my_results20.run

recall_5              	all	0.1214
recall_10             	all	0.1744
recall_15             	all	0.2022
recall_20             	all	0.2262
recall_30             	all	0.2262
recall_100            	all	0.2262
recall_200            	all	0.2262
recall_500            	all	0.2262
recall_1000           	all	0.2262

Trec Eval Evaluation for file my_results30.run

recall_5              	all	0.1214
Trec Eval Evaluation for file my_results50.run


recall_10             	all	0.1744
recall_15             	all	0.2022
recall_20             	all	0.2262
recall_30             	all	0.2584
recall_100            	all	0.2584
recall_200            	all	0.2584
recall_500            	all	0.2584
recall_1000           	all	0.2584
recall_5              	all	0.1214
recall_10             	all	0.1744
recall_15             	all	0.2022
recall_20             	all	0.2262
recall_30             	all	0.2584
recall_100            	all	0.3002
recall_200            	all	0.3002
rec

In [107]:
files = ['my_results20.run','my_results30.run','my_results50.run']
for f in files:
    print(f'Trec Eval Evaluation for file {f}')
    !.\trec_eval\trec_eval -m map trec_eval/test_trec.tsv trec_eval/{f} 2>null
    !.\trec_eval\trec_eval -m P.5 trec_eval/test_trec.tsv trec_eval/{f} 2>null
    !.\trec_eval\trec_eval -m P.10 trec_eval/test_trec.tsv trec_eval/{f} 2>null
    !.\trec_eval\trec_eval -m P.15 trec_eval/test_trec.tsv trec_eval/{f} 2>null
    !.\trec_eval\trec_eval -m P.20 trec_eval/test_trec.tsv trec_eval/{f} 2>null

Trec Eval Evaluation for file my_results20.run
map                   	all	0.1047
P_5                   	all	0.1198
P_10                  	all	0.0860
P_15                  	all	0.0665
P_20                  	all	0.0558
Trec Eval Evaluation for file my_results30.run
map                   	all	0.1076
P_5                   	all	0.1198
P_10                  	all	0.0860
P_15                  	all	0.0665
P_20                  	all	0.0558
Trec Eval Evaluation for file my_results50.run
map                   	all	0.1101
P_5                   	all	0.1198
P_10                  	all	0.0860
P_15                  	all	0.0665
P_20                  	all	0.0558
